In [153]:
import pandas as pd
import numpy as np


In [154]:
data = pd.read_csv('dirty_cafe_sales.csv')
data

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [155]:
df = data.copy()

In [156]:
df.head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


In [157]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [158]:
for col in df.columns:
    print("\n")
    print("====" , col , "====")
    print(df[col].unique())



==== Transaction ID ====
['TXN_1961373' 'TXN_4977031' 'TXN_4271903' ... 'TXN_5255387' 'TXN_7695629'
 'TXN_6170729']


==== Item ====
['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' 'UNKNOWN' 'Sandwich' nan
 'ERROR' 'Juice' 'Tea']


==== Quantity ====
['2' '4' '5' '3' '1' 'ERROR' 'UNKNOWN' nan]


==== Price Per Unit ====
['2.0' '3.0' '1.0' '5.0' '4.0' '1.5' nan 'ERROR' 'UNKNOWN']


==== Total Spent ====
['4.0' '12.0' 'ERROR' '10.0' '20.0' '9.0' '16.0' '15.0' '25.0' '8.0' '5.0'
 '3.0' '6.0' nan 'UNKNOWN' '2.0' '1.0' '7.5' '4.5' '1.5']


==== Payment Method ====
['Credit Card' 'Cash' 'UNKNOWN' 'Digital Wallet' 'ERROR' nan]


==== Location ====
['Takeaway' 'In-store' 'UNKNOWN' nan 'ERROR']


==== Transaction Date ====
['2023-09-08' '2023-05-16' '2023-07-19' '2023-04-27' '2023-06-11'
 '2023-03-31' '2023-10-06' '2023-10-28' '2023-07-28' '2023-12-31'
 '2023-11-07' 'ERROR' '2023-05-03' '2023-06-01' '2023-03-21' '2023-11-15'
 '2023-06-10' '2023-02-24' '2023-03-25' '2023-01-15' '2023-04-04'
 '202

Insights gathered by me : 
- [] things to handle the missing values 
- [] handle UNKNOWN values in columns 
- [] handle ERROR value sin columns 
- [] change the datatypes 
- [] extract the useful columns from existing columns

Understood Insights by claude:

- [] Replace UNKNOWN and ERROR with real NaN, everywhere first
Right now they're just text strings pandas treats as valid values. Until you fix this, every later step (numeric conversion, .isna() counts) will be wrong.

- [] Convert Quantity, Price Per Unit, Total Spent to numeric
They're currently stored as text. Use pd.to_numeric() — but only after step 1, or the leftover "UNKNOWN"/"ERROR" strings will break the conversion.

- [] Recover missing values using the math relationship
Total Spent = Quantity × Price Per Unit holds true in every row that has all three. So wherever exactly one of these three is missing, calculate it from the other two — this recovers 1,398 rows. Don't touch rows missing two or more of these; there's nothing to calculate from.

- [] Fill missing Price Per Unit (and partially Item) using the item-price relationship

    Build a lookup: each item has one fixed price (Coffee=2.0, Cake=3.0, etc.)
    Use it to fill missing Price Per Unit when Item is known
    For filling missing Item from a known price, only do this for the 4 prices that map to exactly one item (1.0, 1.5, 2.0, 5.0) — skip 3.0 and 4.0, since those are shared by two items each and can't be safely guessed

- [] Accept that Payment Method, Location, and Transaction Date are genuinely unrecoverable
Nothing else in the data can tell you these. Leave them as NaN, or drop rows only if your specific analysis absolutely needs that field.

- [] Convert Transaction Date to datetime, extract useful parts
Pull out year/month/day-of-week if useful for your analysis.

- [] Re-verify after every step
Re-run .isna().sum() and spot-check rows after each change, not just once at the end.

In [159]:
df.isna().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

Replace UNKNOWN and ERROR with real NaN, everywhere first

In [160]:
df.isin(['UNKNOWN' , 'ERROR']).sum()

Transaction ID        0
Item                636
Quantity            341
Price Per Unit      354
Total Spent         329
Payment Method      599
Location            696
Transaction Date    301
dtype: int64

In [161]:
df = df.replace(['UNKNOWN', 'ERROR'], np.nan)


In [162]:
df.isin(['UNKNOWN' , 'ERROR']).sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

Convert Quantity, Price Per Unit, Total Spent to numeric

In [163]:
df[['Quantity','Price Per Unit' ,'Total Spent']] = df[['Quantity','Price Per Unit' ,'Total Spent']].apply(pd.to_numeric, errors = 'coerce')

In [164]:
df.dtypes

Transaction ID       object
Item                 object
Quantity            float64
Price Per Unit      float64
Total Spent         float64
Payment Method       object
Location             object
Transaction Date     object
dtype: object

In [165]:
df.isna().sum()

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

In [166]:
df.columns

Index(['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent',
       'Payment Method', 'Location', 'Transaction Date'],
      dtype='object')

In [167]:
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'],errors = 'coerce')

In [168]:
df.dtypes

Transaction ID              object
Item                        object
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
dtype: object

Recover missing values using the math relationship
Total Spent = Quantity × Price Per Unit

In [169]:
df.isna().sum()

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

In [170]:
quality = (df["Quantity"].isna() & df[['Total Spent','Price Per Unit']].notna().all(axis = 1) )
df[quality]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
20,TXN_3522028,Smoothie,NaN,4.0,20.0,Cash,In-store,2023-04-04
55,TXN_5522862,Cookie,NaN,1.0,2.0,Credit Card,Takeaway,2023-03-19
57,TXN_2080895,Cake,NaN,3.0,3.0,Digital Wallet,In-store,2023-04-19
66,TXN_8501819,Juice,NaN,3.0,6.0,Cash,NaN,2023-03-30
117,TXN_2148617,Juice,NaN,3.0,9.0,Digital Wallet,NaN,2023-01-10
...,...,...,...,...,...,...,...,...
9932,TXN_8502079,Tea,NaN,1.5,3.0,Cash,NaN,2023-04-20
9935,TXN_9778251,Tea,NaN,1.5,6.0,NaN,Takeaway,2023-11-09
9944,TXN_7495283,Cake,NaN,3.0,15.0,Credit Card,Takeaway,2023-04-14
9957,TXN_6487003,Coffee,NaN,2.0,8.0,Credit Card,Takeaway,2023-11-15


Quantity = Total Spent / Price Per Unit

In [171]:
df.loc[quality , "Quantity"] = df.loc[quality , "Total Spent"] / df.loc[quality, "Price Per Unit"]


In [172]:
df.isnull().sum()

Transaction ID         0
Item                 969
Quantity              38
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

In [173]:
Price_to_fill = df["Price Per Unit"].isna() & df[["Quantity" , "Total Spent"]].notna().all(axis=1)
df[Price_to_fill]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
56,TXN_3578141,Cake,5.0,NaN,15.0,NaN,Takeaway,2023-06-27
68,TXN_8427104,Salad,2.0,NaN,10.0,NaN,In-store,2023-10-27
85,TXN_8035512,Tea,3.0,NaN,4.5,Cash,NaN,2023-10-29
104,TXN_7447872,Juice,2.0,NaN,6.0,NaN,NaN,NaT
118,TXN_4633784,NaN,5.0,NaN,15.0,NaN,In-store,2023-02-06
...,...,...,...,...,...,...,...,...
9924,TXN_5981429,Juice,2.0,NaN,6.0,Digital Wallet,NaN,2023-12-24
9926,TXN_2464706,Cake,4.0,NaN,12.0,Digital Wallet,Takeaway,2023-11-09
9961,TXN_2153100,Tea,2.0,NaN,3.0,Cash,NaN,2023-12-29
9996,TXN_9659401,NaN,3.0,NaN,3.0,Digital Wallet,NaN,2023-06-02


Price Per Unit = Total Spent / Quantity

In [174]:
df.loc[Price_to_fill,'Price Per Unit'] =  df.loc[Price_to_fill,'Total Spent'] / df.loc[Price_to_fill,'Quantity']
df.isna().sum()
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2.0,2.0,4.0,NaN,NaN,2023-08-30
9996,TXN_9659401,NaN,3.0,1.0,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3.0,1.0,3.0,Digital Wallet,NaN,2023-12-02


In [175]:
total_fill_data = df['Total Spent'].isna() & df[['Quantity' ,'Price Per Unit']].notna().all(axis=1)
df[total_fill_data]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
2,TXN_4271903,Cookie,4.0,1.0,NaN,Credit Card,In-store,2023-07-19
25,TXN_7958992,Smoothie,3.0,4.0,NaN,NaN,NaN,2023-12-13
31,TXN_8927252,NaN,2.0,1.0,NaN,Credit Card,NaN,2023-11-06
42,TXN_6650263,Tea,2.0,1.5,NaN,NaN,Takeaway,2023-01-10
94,TXN_6289610,Juice,3.0,3.0,NaN,Cash,Takeaway,2023-08-07
...,...,...,...,...,...,...,...,...
9890,TXN_2749289,Smoothie,2.0,4.0,NaN,Digital Wallet,Takeaway,2023-05-05
9954,TXN_1191659,Coffee,4.0,2.0,NaN,Credit Card,In-store,2023-11-21
9977,TXN_5548914,Juice,2.0,3.0,NaN,Digital Wallet,In-store,2023-11-04
9988,TXN_9594133,Cake,5.0,3.0,NaN,NaN,NaN,NaT


Total Spent = Quantity * Price Per Unit

In [176]:
df.loc[total_fill_data,'Total Spent'] =  df.loc[total_fill_data,'Quantity'] * df.loc[total_fill_data,'Price Per Unit']
df.isna().sum()

Transaction ID         0
Item                 969
Quantity              38
Price Per Unit        38
Total Spent           40
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

In [177]:
df = df.dropna(subset=["Quantity","Price Per Unit","Total Spent"])
df.isna().sum()

Transaction ID         0
Item                 963
Quantity               0
Price Per Unit         0
Total Spent            0
Payment Method      3158
Location            3940
Transaction Date     457
dtype: int64

In [178]:
look_up = df[['Item' , 'Price Per Unit']].drop_duplicates(subset="Item")
look_up


,Item,Price Per Unit
0,Coffee,2.0
1,Cake,3.0
2,Cookie,1.0
3,Salad,5.0
5,Smoothie,4.0
6,NaN,3.0
7,Sandwich,4.0
17,Juice,3.0
42,Tea,1.5


In [179]:
item = {
   "Coffee" :	2.0,
    "Cake":	3.0,
    "Cookie" : 1.0,
    "Salad" : 5.0,
    "Smoothie" : 4.0,
    "Sandwich" : 4.0,
    "Juice" : 3.0,
    "Tea" : 1.5
}
reverse = {
    price : item
for item, price in item.items()
}

print(reverse)

{2.0: 'Coffee', 3.0: 'Juice', 1.0: 'Cookie', 5.0: 'Salad', 4.0: 'Sandwich', 1.5: 'Tea'}


In [180]:
df['Item'] = df['Item'].fillna(df["Price Per Unit"].map(reverse))

/var/folders/qw/_nk9z1hd74q9vjp80k85r5l80000gn/T/ipykernel_62920/909458718.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Item'] = df['Item'].fillna(df["Price Per Unit"].map(reverse))


In [181]:
df.isna().sum()

Transaction ID         0
Item                   0
Quantity               0
Price Per Unit         0
Total Spent            0
Payment Method      3158
Location            3940
Transaction Date     457
dtype: int64

Accept that Payment Method, Location, and Transaction Date are genuinely unrecoverable
Nothing else in the data can tell you these. Leave them as NaN, or drop rows only if your specific analysis absolutely needs that field.

In [182]:
df['Payment Method'] = df['Payment Method'].fillna("Unknown")
df['Location'] = df['Location'].fillna("Unknown")

/var/folders/qw/_nk9z1hd74q9vjp80k85r5l80000gn/T/ipykernel_62920/1378894148.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Payment Method'] = df['Payment Method'].fillna("Unknown")
/var/folders/qw/_nk9z1hd74q9vjp80k85r5l80000gn/T/ipykernel_62920/1378894148.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Location'] = df['Location'].fillna("Unknown")


In [183]:
df.isna().sum()

Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date    457
dtype: int64

In [184]:
df = df.dropna(subset =["Transaction Date"])

In [185]:
df.isna().sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

In [186]:
df.columns

Index(['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent',
       'Payment Method', 'Location', 'Transaction Date'],
      dtype='object')

In [187]:
df.to_csv('Cleaned_cafe_sales.csv')